# RemediAide 2.0: demo 5 menit

Notebook ini berjalan di Colab gratis, tanpa GPU, tanpa login, dan tanpa data Eedi (lisensinya melarang kami membagikan data olahan).
Isinya empat bagian:

1. Siapkan repo dan paket (sekitar 1 menit).
2. **Lapis 1:** tiga siswa sintetis, aturan CRI v1 dibandingkan model ML yang juga dipakai offline di browser guru.
3. **Lapis 2:** baca alasan siswa dari dataset MAP dan beri nama miskonsepsinya dalam Bahasa Indonesia.
4. **Angka utama:** muat angka test yang tersimpan dan gambar ulang grafiknya. Angka ini **tidak** dihitung ulang, karena test set kami segel dan hanya dibuka sekali (19 November 2026).

Pilih *Runtime → Run all*.

In [ ]:
%%capture
import os, sys, subprocess
if not os.path.exists('remediaide'):
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/muhammadkiro516-netizen/remediaide.git'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'onnxruntime', '-e', 'remediaide/ml'], check=True)

In [ ]:
import sys, json, time
sys.path.insert(0, 'remediaide/ml/src')
import numpy as np, pandas as pd, onnxruntime as ort
ort.set_default_logger_severity(3)
from remediaide_ml.features.history import add_history_features
from remediaide_ml.cri_rule import confidence_to_cri
T0 = time.time()
REPO = 'remediaide'
print('siap')

## Lapis 1: aturan v1 vs model

Tiga siswa sintetis (bukan siswa nyata, bukan data Eedi), masing-masing menjawab 8 soal pada satu konsep:

* **Andi** selalu benar dan yakin.
* **Budi** sering salah tapi tetap yakin. Pola klasik miskonsepsi.
* **Citra** awalnya salah, lalu mulai benar, tapi keyakinannya rendah.

Aturan v1 hanya melihat satu jawaban: benar/salah × yakin/tidak. Model melihat riwayatnya. Model yang dipakai di sini adalah LightGBM fold 0 dalam format ONNX,
model yang sama dengan yang berjalan offline di browser guru. Model utama (CO-KT, ensemble 5 seed) hanya jalan di server.

In [ ]:
def siswa(nama, pola):
    rows = []
    t0 = pd.Timestamp('2026-11-01 08:00')
    for i, (benar, yakin) in enumerate(pola):
        rows.append({'UserId': nama, 'QuestionId': f'q{i}', 'AnswerId': len(rows), 'kc': 'pecahan',
                     'DateAnswered': t0 + pd.Timedelta(minutes=2 * i), 'IsCorrect': benar,
                     'Confidence': yakin, 'has_conf': True, 'cri': confidence_to_cri(yakin), 'attempt_no': 1})
    return rows

data = pd.DataFrame(
    siswa('Andi',  [(1, 100), (1, 100), (1, 75), (1, 100), (1, 100), (1, 75), (1, 100), (1, 100)]) +
    siswa('Budi',  [(0, 100), (0, 75), (1, 75), (0, 100), (0, 100), (1, 100), (0, 75), (0, 100)]) +
    siswa('Citra', [(0, 25), (0, 25), (1, 25), (1, 0), (1, 25), (0, 25), (1, 25), (1, 25)]))
feat = add_history_features(data)
meta = json.load(open(f'{REPO}/public/models/l1_features.json'))
for k, v in meta['defaults'].items():
    feat[k] = v   # soal sintetis belum punya statistik kesulitan dari data latih
X = feat[meta['features']].astype('float32').to_numpy()

sess = ort.InferenceSession(f'{REPO}/public/models/l1_b4.onnx')
p4 = sess.run(None, {sess.get_inputs()[0].name: X})[1]
p4 = np.array([[r[i] for i in range(4)] for r in p4]) if isinstance(p4, list) else np.asarray(p4)
NAMA = {'P': 'Paham', 'L': 'Tebakan beruntung', 'M': 'Miskonsepsi', 'T': 'Tidak paham'}
kelas = meta['classes_b4']
hasil = feat[['UserId', 'IsCorrect', 'Confidence']].copy()
hasil['aturan v1'] = feat['quadrant_b0'].map(NAMA)
hasil['model'] = [NAMA[kelas[i]] for i in p4.argmax(1)]
hasil['yakin model'] = p4.max(1).round(2)
hasil.groupby('UserId').tail(3)

Lihat baris Budi yang **benar dan yakin**. Aturan v1 langsung menilainya "Paham", karena hanya melihat jawaban itu. Model menilainya
"Miskonsepsi", karena Budi sudah berkali-kali salah dengan yakin di konsep yang sama, jadi satu jawaban benar belum cukup. Citra kebalikannya: dia mulai benar tapi
tidak yakin, dan aturan v1 menyebutnya "Tebakan beruntung" tanpa melihat bahwa sebelumnya dia selalu salah.

Ini contoh sintetis untuk menunjukkan cara kerja, bukan bukti. Buktinya ada di bagian terakhir.

In [ ]:
cfg = json.load(open(f'{REPO}/ml/models/l3_conformal_v1.json'))
ragu = 1 - p4.max(1)
hasil['serahkan ke guru'] = ragu >= cfg['l1_doubt_threshold']
print('Ambang ragu (dari kalibrasi CO-KT):', round(cfg['l1_doubt_threshold'], 3))
print('Catatan: ambang ini dikalibrasi untuk CO-KT; di sini dipakai pada model browser hanya sebagai ilustrasi.')
hasil.groupby('UserId').tail(1)[['UserId', 'aturan v1', 'model', 'yakin model', 'serahkan ke guru']]

## Lapis 2: membaca alasan siswa

Kami unduh dataset MAP (36.696 alasan siswa, lisensi MIT) dari Hugging Face, lalu melatih versi cepat baseline TF-IDF.
Versi ini memakai `SGDClassifier`, bukan regresi logistik produksi, supaya selesai dalam hitungan detik. Nama miskonsepsi diterjemahkan dengan kamus Indonesia kami
(39 pasangan soal-miskonsepsi dari Tabel 1 Rittle-Johnson dkk. 2025).

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import SGDClassifier
from sklearn.model_selection import train_test_split
try:
    mp = pd.read_csv('https://huggingface.co/datasets/kaggle-map/data/resolve/main/train.csv')
except Exception as e:
    raise SystemExit(f'Gagal mengunduh MAP dari Hugging Face: {e}')
mp['target'] = mp['Category'] + ':' + mp['Misconception'].fillna('NA')
mp['teks'] = 'Q' + mp['QuestionId'].astype(str) + ' ' + mp['MC_Answer'].astype(str) + ' || ' + mp['StudentExplanation'].astype(str)
tr, te = train_test_split(mp, test_size=0.2, random_state=20261118)
vec = TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)
clf = SGDClassifier(loss='log_loss', alpha=1e-5, max_iter=30, random_state=0).fit(vec.fit_transform(tr['teks']), tr['target'])
P = clf.predict_proba(vec.transform(te['teks']))
top3 = clf.classes_[np.argsort(-P, 1)[:, :3]]
map3 = np.mean([(1.0 if t == a[0] else 0.5 if t == a[1] else 1/3 if t == a[2] else 0.0) for t, a in zip(te['target'], top3)])
print(f'MAP@3 versi cepat (split acak 80/20): {map3:.3f}')

In [ ]:
kamus = json.load(open(f'{REPO}/ml/models/misconception_dictionary.json'))
contoh = te[te['Misconception'].notna()].groupby('Misconception').head(1).head(3)
for _, r in contoh.iterrows():
    p = clf.predict_proba(vec.transform([r['teks']]))[0]
    lab = clf.classes_[p.argmax()]
    misc = lab.split(':')[1]
    print('Alasan siswa :', r['StudentExplanation'][:120])
    print('Prediksi     :', lab, f'(p = {p.max():.2f})')
    print('Artinya      :', kamus.get(misc, '(bukan miskonsepsi)'))
    print('Label asli   :', r['target'])
    print()

## Angka utama dari test yang disegel

Angka di bawah dibaca dari `ml/reports/tables/headline.json`, hasil satu kali pembukaan test (773 siswa Eedi yang tidak pernah dipakai melatih atau memilih model).
Notebook ini tidak menghitung ulang angka tersebut.

In [ ]:
import matplotlib.pyplot as plt
h = json.load(open(f'{REPO}/ml/reports/tables/headline.json'))
lab = ['Aturan CRI v1', 'Regresi logistik\n(kesulitan soal)', 'Regresi logistik\n(+ riwayat)', 'RemediAide 2.0', 'RemediAide 2.0\nsetelah defer']
val = [h['b0_misgroup'], h['b1a_misgroup'], h['b1b_misgroup'], h['ml_misgroup'], h['ml_misgroup_kept']]
fig, ax = plt.subplots(figsize=(8, 3.6))
bars = ax.bar(lab, [v * 100 for v in val], color=['#9aa0a6', '#c4c7ca', '#c4c7ca', '#1a73e8', '#0b57d0'])
for b, v in zip(bars, val):
    ax.text(b.get_x() + b.get_width() / 2, v * 100 + 0.8, f'{v*100:.1f}%', ha='center', fontsize=10)
ax.set_ylabel('salah kelompok (%)'); ax.set_ylim(0, 55); ax.spines[['top', 'right']].set_visible(False)
ax.set_title('Test: 58.963 kasus, 773 siswa (lebih rendah lebih baik)', fontsize=11)
plt.tight_layout(); plt.show()
print(f"Turun {-h['abs_diff_points']:.1f} poin [{-h['abs_diff_points_ci'][1]:.1f}; {-h['abs_diff_points_ci'][0]:.1f}] dibanding aturan v1.")
print(f"Lebih baik {-h['cokt_minus_b1b_points']:.1f} poin dari baseline sederhana terkuat.")
print(f"{h['defer_rate']*100:.1f}% kasus paling ragu diserahkan ke guru.")
print(f'Waktu total notebook: {time.time() - T0:.0f} detik')

Yang belum berhasil (recall Miskonsepsi di test, kesenjangan fairness, label proxy, terjemahan mesin) ditulis terbuka di README dan
[`ml/REPORT.md`](https://github.com/muhammadkiro516-netizen/remediaide/blob/main/ml/REPORT.md).